In [ ]:
import csv
import json
import os
import random
import re
from copy import deepcopy
from typing import Callable, Optional, Any

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Числа: поддерживаются 1,200 / 1,200.50 / 4.0 / -3; число в конце предложения ("= 18.") тоже находится.
NUMBER_PATTERN = re.compile(
    r"(?<![\w.,])-?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?!\w)"
    r"|(?<![\w.,])-?\d+(?:\.\d+)?(?!\w)"
)
# Нумерация шага ("Step 2:", "3. ") - числа в ней не трогаем.
STEP_LABEL = re.compile(r"^\s*(?:step\s*\d+\s*[:.)-]?\s*|\d+[.)]\s+)", re.I)


def find_numbers(step):
    label = STEP_LABEL.match(step)
    start = label.end() if label else 0
    return [m for m in NUMBER_PATTERN.finditer(step) if m.start() >= start]


def extract_number_ex(text):
    """Возвращает (ответ, format_ok). format_ok=False, если формат "Answer:" не соблюдён
    и взято последнее число в тексте (ненадёжно)."""
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s, ok = m.group(1), True
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s, ok = (nums[-1] if nums else None), False
    if not s:
        return None, False
    return s.replace(",", "").rstrip("."), ok


def extract_letter_ex(text):
    """Возвращает (буква, format_ok). Запасной поиск устойчив к артиклю "A" в тексте."""
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1), True
    cands = re.findall(r"(?i:answer\s+is)\s*:?\s*\(?([A-E])\b", text)
    if not cands:
        cands = re.findall(r"\(([A-E])\)", text)
    return (cands[-1], False) if cands else (None, False)


def extract_number(text):
    return extract_number_ex(text)[0]


def extract_letter(text):
    return extract_letter_ex(text)[0]


def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False


def split_steps(text):
    lines = [s.strip() for s in text.split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]

In [ ]:
def parse_steps(value):
    """
    Читает столбец steps из CSV.
    В ноутбуке он записывается как JSON-список строк.
    Если JSON некорректен, использует split_steps().
    """
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    if value is None:
        return []

    value = str(value).strip()
    if not value:
        return []

    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except json.JSONDecodeError:
        pass

    return split_steps(value)


def serialize_steps(steps):
    return json.dumps(steps, ensure_ascii=False)


def normalize_dataset(dataset):
    """Приводит обозначения датасетов к двум поддерживаемым вариантам."""
    value = str(dataset).strip().lower()
    if value in {"gsm8k", "gsm-8k"}:
        return "gsm8k"
    if value.startswith("arc"):
        return "arc"
    return value

def extract_answer_ex(text, dataset):
    """(ответ, format_ok) тем же парсером, что и в generation.ipynb (кроме исправленного запасного пути)."""
    dataset = normalize_dataset(dataset)
    if dataset == "gsm8k":
        return extract_number_ex(text)
    return extract_letter_ex(text)


def extract_answer(text, dataset):
    return extract_answer_ex(text, dataset)[0]


def answers_equal(prediction, gold, dataset):
    """Проверка ответа с учётом типа датасета."""
    dataset = normalize_dataset(dataset)
    if prediction is None or gold is None:
        return False
    if dataset == "gsm8k":
        return same_number(prediction, gold)
    return str(prediction).strip().upper() == str(gold).strip().upper()

In [ ]:
def _format_like(old_text, value):
    """Форматирует новое число так же, как старое (знаки после точки, разделитель тысяч)."""
    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    return f"{value:,.{decimals}f}" if "," in old_text else f"{value:.{decimals}f}"


def corrupt_value(old_value, decimals, rng):
    """
    Правдоподобная подмена: сдвиг на 10 / 25 / 50 % в случайную сторону
    (минимум на единицу последнего разряда). Знак сохраняется, ноль не получается.
    Для нуля берётся маленькое положительное число.
    """
    unit = 10 ** (-decimals)
    if old_value == 0:
        return rng.choice([1, 2, 3, 5]) * unit
    for _ in range(30):
        delta = rng.choice([-1, 1]) * max(unit, abs(old_value) * rng.choice([0.1, 0.25, 0.5]))
        new_value = round(old_value + delta, decimals)
        if new_value != old_value and new_value * old_value > 0:
            return new_value
    return None


def replace_number(steps, step_index, number_index, rng):
    """
    Заменяет одно число в выбранном шаге.
    Возвращает (изменённые_шаги, исходное_число, новое_число, относительный_сдвиг) или None.
    """
    if not 0 <= step_index < len(steps):
        return None

    matches = find_numbers(steps[step_index])
    if not 0 <= number_index < len(matches):
        return None

    match = matches[number_index]
    old_text = match.group()

    try:
        old_value = float(old_text.replace(",", ""))
    except ValueError:
        return None

    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    new_value = corrupt_value(old_value, decimals, rng)
    if new_value is None:
        return None

    new_text = _format_like(old_text, new_value)
    if new_text == old_text:
        return None

    modified = deepcopy(steps)
    modified[step_index] = (
        modified[step_index][:match.start()]
        + new_text
        + modified[step_index][match.end():]
    )
    relative_shift = (new_value - old_value) / old_value if old_value != 0 else None
    return modified, old_text, new_text, relative_shift


def truncate_steps(steps, keep_count):
    """Оставляет первые keep_count шагов (0 - рассуждения нет вообще)."""
    if not 0 <= keep_count < len(steps):
        raise ValueError("keep_count должен быть от 0 до len(steps)-1")
    return deepcopy(steps[:keep_count])


def swap_adjacent(steps, i):
    """Меняет местами шаги i и i+1. None, если шаги одинаковые (подмена была бы пустой)."""
    if not 0 <= i < len(steps) - 1 or steps[i] == steps[i + 1]:
        return None
    modified = deepcopy(steps)
    modified[i], modified[i + 1] = modified[i + 1], modified[i]
    return modified


def distinct_permutations(steps, rng, n_perms=3, max_tries=50):
    """До n_perms различных перестановок, каждая заведомо отличается от исходного порядка."""
    seen = {tuple(steps)}
    out = []
    for _ in range(max_tries):
        if len(out) >= n_perms:
            break
        candidate = list(steps)
        rng.shuffle(candidate)
        if tuple(candidate) not in seen:
            seen.add(tuple(candidate))
            out.append(candidate)
    return out

In [ ]:
ANSWER_CUE = "Answer:"
ANSWER_MAX_TOKENS = 24      # режим "answer": модель пишет только сам ответ
CONTINUE_MAX_TOKENS = 700   # режим "continue": модель дописывает цепочку (как в generation.ipynb)


def question_numbers(prompt):
    """Числа из условия (текст после последнего "Question:"; эвристика для промптов few-shot)."""
    parts = re.split(r"(?i)question\s*:", prompt or "")
    text = parts[-1] if len(parts) > 1 else (prompt or "")
    values = set()
    for m in NUMBER_PATTERN.finditer(text):
        try:
            values.add(float(m.group().replace(",", "")))
        except ValueError:
            pass
    return values


def _rel_pos(idx, n):
    return idx / (n - 1) if n > 1 else 0.0


def build_variants(steps, rng, given_numbers=frozenset(), max_numbers=None, n_permutations=3):
    """
    Варианты вмешательств. resume:
      "answer"   - после (изменённой) цепочки принудительно пишем "Answer:" и генерируем только ответ;
      "continue" - модель сама дописывает цепочку после изменённого префикса
                   (для подмены числа префикс обрывается на изменённом шаге).
    Контроли: "control" (цепочка не тронута, режим answer) и "control_continue"
    (тот же префикс без подмены, режим continue) - без них answer_changed нельзя интерпретировать.
    """
    n = len(steps)
    variants = []

    def add(perturbation, resume, modified, idx=None, rel_pos=None, kept=None,
            num_idx=None, old=None, new=None, shift=None, given=None, last=None):
        variants.append({
            "perturbation": perturbation, "resume": resume,
            "target_step_index": idx, "relative_position": rel_pos, "kept_fraction": kept,
            "target_number_index": num_idx, "old_number": old, "new_number": new,
            "relative_shift": shift, "number_is_given": given, "is_last_number_in_step": last,
            "modified_steps": modified,
        })

    # Контроли
    add("control", "answer", deepcopy(steps))
    for k in range(n):
        add("control_continue", "continue", deepcopy(steps[:k + 1]), idx=k, rel_pos=_rel_pos(k, n))

    # Подмена чисел: одно и то же новое число проверяется в обоих режимах (парное сравнение)
    targets = [(si, ni) for si, step in enumerate(steps) for ni in range(len(find_numbers(step)))]
    if max_numbers is not None and len(targets) > max_numbers:
        targets = sorted(rng.sample(targets, max_numbers))
    for si, ni in targets:
        result = replace_number(steps, si, ni, rng)
        if result is None:
            continue
        modified, old_number, new_number, shift = result
        old_value = float(old_number.replace(",", ""))
        given = old_value in given_numbers
        last = ni == len(find_numbers(steps[si])) - 1
        for resume in ("answer", "continue"):
            add("number_substitution", resume,
                modified if resume == "answer" else modified[:si + 1],
                idx=si, rel_pos=_rel_pos(si, n), num_idx=ni,
                old=old_number, new=new_number, shift=shift, given=given, last=last)

    # Обрезка: от 0 (без рассуждения) до n-1 шагов
    for keep in range(n):
        add("truncation", "answer", truncate_steps(steps, keep), idx=keep, kept=keep / n)

    # Перестановка соседних шагов (зависимость от позиции) и полные перестановки
    for i in range(n - 1):
        swapped = swap_adjacent(steps, i)
        if swapped is not None:
            add("adjacent_swap", "answer", swapped, idx=i, rel_pos=_rel_pos(i, n))
    for perm in distinct_permutations(steps, rng, n_permutations):
        add("step_permutation", "answer", perm)

    return variants


def build_assistant_prefix(modified_steps, resume):
    """
    Цепочка подаётся как начало ответа ассистента (как будто модель написала её сама),
    а не как чужой текст в сообщении пользователя.
    """
    chain = "\n\n".join(modified_steps)
    if resume == "answer":
        return (chain + "\n\n" + ANSWER_CUE) if chain else ANSWER_CUE
    return chain + "\n\n"

In [ ]:
FIELDNAMES = [
    "question_id", "dataset", "model", "mode",
    "perturbation_id", "perturbation", "resume", "n_steps",
    "target_step_index", "relative_position", "kept_fraction",
    "target_number_index", "old_number", "new_number", "relative_shift",
    "number_is_given", "is_last_number_in_step",
    "original_steps", "perturbed_steps", "assistant_prefix",
    "original_answer", "gold_answer",
    "regenerated_text", "regenerated_answer", "format_ok",
    "original_correct", "answer_changed", "regenerated_correct",
]


def _row_key(row):
    return (str(row.get("question_id", "")), str(row.get("model", "")), str(row.get("mode", "")))


def _truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes"}


def run_experiment(
    input_csv,
    output_csv,
    generate_fn: Callable,
    seed=42,
    max_rows: Optional[int] = None,
    save_every: int = 10,
    resume: bool = False,
    skip_truncated: bool = True,
    only_correct: bool = True,
    max_numbers: Optional[int] = None,
    n_permutations: int = 3,
):
    """
    Запуск вмешательств.

    generate_fn(prompt, row, assistant_prefix, max_new_tokens) -> generated_text
        prompt - исходный prompt из CSV без изменений; assistant_prefix - начало ответа
        ассистента (цепочка), которое модель должна продолжить.

    save_every: после каждых save_every задач (вопрос со всеми вариантами) результаты
        дописываются в output_csv и сбрасываются на диск.
    resume: True - пропустить уже сохранённые вопросы и дописывать файл (схема столбцов
        должна совпадать); False - создать файл заново.
    skip_truncated: пропускать цепочки, обрезанные по лимиту токенов (столбец truncated).
    only_correct: брать только изначально верные ответы - тогда answer_changed значит
        "подмена разрушила верный ответ", а не "неверный ответ был случайным".
    max_numbers: максимум подменяемых чисел на вопрос (None - все); экономит время генерации.
    n_permutations: число случайных полных перестановок на вопрос.

    Извлечение ответа и проверка корректности используют парсеры generation.ipynb.
    """
    rng = random.Random(seed)

    # 1. Планирование: варианты строятся для всех подходящих строк по порядку,
    # поэтому подмены не зависят от resume.
    tasks = []
    with open(input_csv, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row_number, row in enumerate(reader):
            if max_rows is not None and row_number >= max_rows:
                break

            original_steps = parse_steps(row.get("steps", ""))
            if len(original_steps) < 2:      # direct-ответы (steps=[]) - контрольная группа
                continue
            if skip_truncated and _truthy(row.get("truncated", "")):
                continue
            if only_correct and not _truthy(row.get("correct", "")):
                continue

            variants = build_variants(
                original_steps, rng,
                given_numbers=question_numbers(row.get("prompt", "")),
                max_numbers=max_numbers, n_permutations=n_permutations,
            )
            tasks.append((row, original_steps, variants))

    # 2. Что уже сохранено (для resume).
    append = resume and os.path.exists(output_csv) and os.path.getsize(output_csv) > 0
    done = set()
    if append:
        with open(output_csv, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames != FIELDNAMES:
                raise ValueError(
                    "Схема столбцов в существующем файле не совпадает с текущей версией. "
                    "Укажите новый output_csv или resume=False."
                )
            done = {_row_key(r) for r in reader}

    todo = [t for t in tasks if _row_key(t[0]) not in done]
    total_generations = sum(len(variants) for _, _, variants in todo)

    print(
        f"Задач с цепочкой: {len(tasks)}, уже сохранено: {len(tasks) - len(todo)}, "
        f"к обработке: {len(todo)} (генераций: {total_generations})"
    )

    results = []
    pending = []
    saved_rows = 0

    out = open(
        output_csv,
        "a" if append else "w",
        encoding="utf-8" if append else "utf-8-sig",
        newline="",
    )
    writer = csv.DictWriter(out, fieldnames=FIELDNAMES)
    if not append:
        writer.writeheader()
        out.flush()

    def flush(bar):
        nonlocal saved_rows
        if not pending:
            return
        writer.writerows(pending)
        out.flush()
        os.fsync(out.fileno())
        saved_rows += len(pending)
        bar.write(f"[сохранено] всего строк в файле от этого запуска: {saved_rows}")
        pending.clear()

    bar = tqdm(total=total_generations, desc="Генерация", unit="ген")

    try:
        for task_idx, (row, original_steps, variants) in enumerate(todo, start=1):
            dataset = normalize_dataset(row["dataset"])
            question_results = []

            original_answer = row.get("extracted_answer", "")
            gold_answer = row.get("gold_answer", "")
            original_missing = original_answer is None or str(original_answer).strip() == ""

            for variant_id, variant in enumerate(variants):
                modified_steps = variant["modified_steps"]
                mode_resume = variant["resume"]
                assistant_prefix = build_assistant_prefix(modified_steps, mode_resume)
                max_new = ANSWER_MAX_TOKENS if mode_resume == "answer" else CONTINUE_MAX_TOKENS

                generated_text = generate_fn(row.get("prompt", ""), row, assistant_prefix, max_new)

                # В режиме answer префикс заканчивается на "Answer:", поэтому дописываем его для парсинга.
                parse_text = (ANSWER_CUE + generated_text) if mode_resume == "answer" else generated_text
                new_answer, format_ok = extract_answer_ex(parse_text, dataset)

                # Пустой/неизвлечённый ответ (в том числе исходный) не считаем изменением ответа.
                if new_answer is None or str(new_answer).strip() == "" or original_missing:
                    answer_changed = None
                else:
                    answer_changed = not answers_equal(new_answer, original_answer, dataset)

                regenerated_correct = (
                    answers_equal(new_answer, gold_answer, dataset)
                    if new_answer is not None
                    else False
                )

                question_results.append({
                    "question_id": row.get("question_id", ""),
                    "dataset": dataset,
                    "model": row.get("model", ""),
                    "mode": row.get("mode", ""),
                    "perturbation_id": variant_id,
                    "perturbation": variant["perturbation"],
                    "resume": mode_resume,
                    "n_steps": len(original_steps),
                    "target_step_index": variant["target_step_index"],
                    "relative_position": variant["relative_position"],
                    "kept_fraction": variant["kept_fraction"],
                    "target_number_index": variant["target_number_index"],
                    "old_number": variant["old_number"],
                    "new_number": variant["new_number"],
                    "relative_shift": variant["relative_shift"],
                    "number_is_given": variant["number_is_given"],
                    "is_last_number_in_step": variant["is_last_number_in_step"],
                    "original_steps": serialize_steps(original_steps),
                    "perturbed_steps": serialize_steps(modified_steps),
                    "assistant_prefix": assistant_prefix,
                    "original_answer": original_answer,
                    "gold_answer": gold_answer,
                    "regenerated_text": generated_text,
                    "regenerated_answer": new_answer,
                    "format_ok": format_ok,
                    "original_correct": _truthy(row.get("correct", "")),
                    "answer_changed": answer_changed,
                    "regenerated_correct": regenerated_correct,
                })

                bar.update(1)
                bar.set_postfix(
                    задача=f"{task_idx}/{len(todo)}",
                    q=row.get("question_id", ""),
                    сохранено=saved_rows,
                )

            # Вопрос попадает в буфер только целиком
            pending.extend(question_results)
            results.extend(question_results)

            if task_idx % save_every == 0:
                flush(bar)
    finally:
        # Сохраняем завершённые вопросы даже при прерывании
        flush(bar)
        bar.close()
        out.close()

    print(f"Эксперимент завершён. Вариантов в этом запуске: {len(results)}")
    print(f"Результаты сохранены в {output_csv}")
    return results

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Устройство:", model.device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Устройство: cuda:0


In [ ]:
def generate(model, tokenizer, user_prompt, max_new_tokens, assistant_prefix=""):
    """assistant_prefix - начало ответа ассистента, которое модель продолжает."""
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text += assistant_prefix
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [ ]:
def qwen_generate(prompt, row, assistant_prefix="", max_new_tokens=700):
    generated_text, n_tokens = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        assistant_prefix=assistant_prefix,
    )
    return generated_text

In [ ]:
INPUT_CSV = "/content/results_test100_Qwen2.5-0.5B-Instruct.csv"
OUTPUT_CSV = "/content/output/perturbation_results_0.5B.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

run_experiment(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV,
    generate_fn=qwen_generate,
    seed=42,
    save_every=10,
    resume=False,        # True - продолжить после падения, не пересчитывая сохранённые вопросы
    skip_truncated=True, # не брать цепочки, обрезанные по лимиту токенов
    only_correct=True,   # брать только изначально верные ответы
    max_numbers=None,    # например 12 - ограничить число подменяемых чисел на вопрос
)

In [ ]:
import numpy as np
import pandas as pd


def load_results(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    for col in ("answer_changed", "regenerated_correct", "format_ok", "original_correct", "number_is_given", "is_last_number_in_step"):
        df[col] = df[col].map(lambda v: np.nan if pd.isna(v) else float(str(v) == "True"))
    # общая позиция: для обрезки - доля сохранённых шагов, для остальных - позиция изменённого шага
    df["pos"] = df["relative_position"].fillna(df["kept_fraction"])
    df["pos_bin"] = pd.cut(df["pos"], [-0.001, 1 / 3, 2 / 3, 1.0], labels=["начало", "середина", "конец"])
    return df


def bootstrap_mean(values, n_boot=2000, seed=0):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x))).mean(axis=1)
    return x.mean(), *np.percentile(means, [2.5, 97.5])


def summarize(df, by, value="answer_changed"):
    """
    Сначала усредняем внутри вопроса (длинные цепочки не перевешивают),
    затем бутстрап ПО ВОПРОСАМ, а не по строкам - варианты одного вопроса зависимы.
    """
    group_cols = list(dict.fromkeys(list(by) + ["question_id", "model", "mode"]))
    per_question = df.groupby(group_cols, dropna=False, observed=True)[value].mean().reset_index()
    rows = []
    for key, g in per_question.groupby(list(by), dropna=False, observed=True):
        mean, lo, hi = bootstrap_mean(g[value])
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(by, key)), "n_questions": g[value].notna().sum(),
                     value: mean, "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


df = load_results(OUTPUT_CSV)
print("строк:", len(df), "| вопросов:", df[["question_id", "model", "mode"]].drop_duplicates().shape[0])
print("доля format_ok=False (ответ взят запасным путём):", round((df["format_ok"] == 0).mean(), 3))
df.groupby(["perturbation", "resume"]).size().rename("variants")

In [ ]:
# Основная таблица: доля изменённых ответов по видам вмешательств. Сравнивайте с control / control_continue.
valid = df[df["format_ok"] == 1]          # только ответы в формате "Answer:"
summarize(valid, ["dataset", "mode", "perturbation", "resume"])

In [ ]:
# Зависимость от позиции: control_continue - базовый уровень для number_substitution (resume=continue)
position_table = summarize(
    valid[valid["perturbation"].isin(["control_continue", "number_substitution", "adjacent_swap", "truncation"])],
    ["dataset", "perturbation", "resume", "pos_bin"],
)
position_table

In [ ]:
# Подмена чисел: исходное данное из условия или промежуточный результат; последнее число шага или нет
numsub = valid[valid["perturbation"] == "number_substitution"]
summarize(numsub, ["resume", "number_is_given", "is_last_number_in_step"])